# Giải thích Code Question 1: Reflex Agent

Trong câu hỏi này, chúng ta cần tạo ra một **Hàm đánh giá (Evaluation Function)** cho Pac-Man. 

## Tổng quan về Hàm Evaluation Function

**1. Mục đích của hàm này là gì?**
Hãy tưởng tượng bạn đang chơi cờ vua. Trước khi đi một nước cờ, bạn sẽ nhìn vào bàn cờ và tưởng tượng: *"Nếu mình đi nước này, thì thế cờ tiếp theo sẽ trông như thế nào? Nó tốt hay xấu?"*. Hàm đánh giá của chúng ta cũng hoạt động y hệt như vậy! Nó giúp "não bộ" của Pac-Man đánh giá xem một hành động (như đi Lên, Xuống, Trái, Phải) sẽ dẫn tới kết quả tương lai tốt hay xấu, từ đó chọn ra hành động có lợi nhất.

**2. Nó nhận vào tham số gì?**
Hàm `evaluationFunction(self, currentGameState, action)` nhận vào 2 tham số chính:
*   `currentGameState` (Trạng thái hiện tại): Bức tranh toàn cảnh của bàn cờ lúc này (Pac-Man đang ở đâu, ma ở đâu, thức ăn ở đâu...).
*   `action` (Hành động): Hành động mà Pac-Man đang *dự định* thực hiện (ví dụ: `Directions.NORTH` - đi lên).

Từ 2 tham số này, hệ thống (ở dòng code đầu tiên) sẽ tự động tạo ra một `successorGameState` (Trạng thái kế tiếp), tức là "viễn cảnh tương lai" mô phỏng chính xác xem nếu Pac-Man thực sự đi bước đó thì bàn cờ sẽ thay đổi ra sao.

**3. Nó trả về cái gì?**
Hàm này bắt buộc phải trả về một **con số (number)**. 
*   Số càng **lớn**, nghĩa là viễn cảnh tương lai đó càng **tốt** (ví dụ: được cộng điểm, tới gần thức ăn).
*   Số càng **nhỏ** hoặc thậm chí là âm vô cực, nghĩa là viễn cảnh đó rất **xấu** (ví dụ: bị ma cắn).

Dựa vào các con số được trả về cho từng hướng đi, Pac-Man sẽ chỉ đơn giản là chọn hướng đi mang lại điểm số cao nhất!

---

Mục tiêu cốt lõi của hàm đánh giá mà chúng ta sắp viết dưới đây gồm 2 ý: **Tránh ma để sống sót** và **Tiến tới thức ăn gần nhất để ghi điểm**.

Dưới đây là phần giải thích chi tiết từng dòng code đã được thêm vào:

## 1. Tránh Ma

In [ ]:
# Check for active ghosts nearby
for ghostState in newGhostStates:
    ghostPos = ghostState.getPosition()
    # If a ghost is active (not scared) and is too close, this is a very bad state
    if ghostState.scaredTimer == 0 and manhattanDistance(newPos, ghostPos) <= 1:
        return -float('inf')

*   **Ý nghĩa:** Sinh tồn là ưu tiên số một. Nếu Pac-Man đi vào một vị trí mà ngay cạnh nó, có nghĩa là khoảng cách bằng có một con ma đang săn mồi, không bị sợ do Pac-Man chưa ăn viên sức mạnh, thì đó là một bước đi **cực kỳ tồi tệ**.
*   **Giải thích code:** 
    *   Chúng ta duyệt qua danh sách các con ma (`newGhostStates`).
    *   Lấy vị trí của từng con ma (`ghostPos`).
    *   Kiểm tra hai điều kiện: (1) Con ma không bị sợ (`scaredTimer == 0`) and (2) Khoảng cách từ Pac-Man đến con ma nhỏ hoặc bằng 1 (`manhattanDistance <= 1`). Khoảng cách Manhattan là cách tính đường đi theo lưới, giống như đi trên đường phố bàn cờ vuông vức.
    *   Nếu cả hai điều kiện đúng, hàm lập tức trả về `-float('inf')`, có nghĩa là âm vô cùng. Điều này đánh dấu bước đi này là "chắc chắn chết, không bao giờ được chọn".

## 2. Tìm Thức Ăn

In [ ]:
# Find distance to closest food
foodList = newFood.asList()
minFoodDist = float('inf')
for foodPos in foodList:
    dist = manhattanDistance(newPos, foodPos)
    if dist < minFoodDist:
        minFoodDist = dist

*   **Ý nghĩa:** Nếu bước đi an toàn, Pac-Man cần biết nó nên đi đâu tiếp. Cách tốt nhất là đi về phía có thức ăn gần nhất để ăn nó nhanh nhất có thể.
*   **Giải thích code:**
    *   Lấy danh sách tọa độ tất cả thức ăn còn lại trên bản đồ (`foodList`).
    *   Đặt một biến `minFoodDist` bằng vô cùng ban đầu để đảm bảo bất kỳ khoảng cách thực tế nào cũng nhỏ hơn nó.
    *   Duyệt qua từng viên thức ăn, tính khoảng cách từ Pac-Man tới viên đó (`dist`).
    *   So sánh và cập nhật để tìm ra khoảng cách tới viên thức ăn **gần nhất** (`minFoodDist`).

## 3. Tính Tổng Điểm

In [ ]:
score = successorGameState.getScore()
if minFoodDist != float('inf'):
    score += 1.0 / minFoodDist
    
return score

*   **Ý nghĩa:** Kết hợp các yếu tố lại để cho ra một con số tổng quát để so sánh với các hướng đi khác.
*   **Giải thích code:**
    *   `successorGameState.getScore()`: Đây là điểm số *có sẵn* của trò chơi nếu Pac-Man thực hiện bước đi này (ví dụ: tới nơi ăn được thức ăn thì game tự cộng 10 điểm). Chúng ta lấy điểm số này làm nền tảng.
    *   `1.0 / minFoodDist`: Đây là bonus để khuyến khích Pac-Man. Tại sao lại lấy `1` chia cho khoảng cách? 
        *   Nếu khoảng cách càng nhỏ, thức ăn càng gần thì phân số này càng lớn, phần thưởng càng to. 
        *   Nếu khoảng cách càng lớn, thức ăn càng xa thì phân số này càng nhỏ.
        *   Nhờ phân số này, nếu có 2 hướng đi an toàn và đều chưa ăn được thức ăn ngay lập tức, Pac-Man sẽ ưu tiên hướng đi nào làm cho nó gần viên thức ăn nhất.
    *   Cuối cùng, trả về tổng điểm (`score`).

Tóm lại, ở mỗi ngã rẽ, Pac-Man tưởng tượng thử từng hướng đi (Trái, Phải, Lên, Xuống). Đối với mỗi hướng, nó chấm điểm bằng đoạn code trên. Nếu hướng nào bị ma cắn, điểm là âm vô cực. Các hướng còn lại, hướng nào điểm cao nhất thì nó sẽ quyết định đi theo hướng đó!